# IT4060 - HPC Failure Prediction

## Notebook: Feature Engineering

This notebook loads the cleaned interim tables, aggregates node-hour features, creates lag and rolling signals, adds multi-horizon labels (`next 1h`, `6h`, `12h`, `24h`), and saves the final modeling dataset to `data/processed/`.

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

In [8]:
PROJECT_NAME = 'IT4060-ML-Assignment-HPC-Failure-Prediction'

def find_project_root():
    cwd = Path.cwd().resolve()
    home = Path.home().resolve()
    desktop = home / 'Desktop'
    candidate_roots = [cwd, *cwd.parents, home, desktop, desktop / 'Manilka' / 'ML_Assignment']
    seen = set()

    for base in candidate_roots:
        for candidate in (base, base / PROJECT_NAME):
            if candidate in seen or not candidate.exists():
                continue
            seen.add(candidate)
            if (candidate / 'data' / 'interim').exists():
                return candidate

    raise FileNotFoundError('Could not locate the project root with data/interim.')

project_root = find_project_root()
interim_dir = project_root / 'data' / 'interim'
processed_dir = project_root / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

failure_clean_path = interim_dir / 'failure_system20_clean.csv'
usage_nodes_clean_path = interim_dir / 'usage_node_events_clean.csv.gz'
feature_export_path = processed_dir / 'node_hour_features_multi_horizon.csv.gz'

required_files = [failure_clean_path, usage_nodes_clean_path]
missing_files = [path.name for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        'Missing cleaned interim files: ' + ', '.join(missing_files) + '. Run 02_data_cleaning.ipynb first.'
    )

print(f'Working directory: {Path.cwd()}')
print(f'Project root: {project_root}')
print(f'Interim directory: {interim_dir}')
print(f'Processed directory: {processed_dir}')

Working directory: d:\_4TH_Year_2nd_Semester_Stuff\ML\Assignments\_Assignment\IT4060-ML-Assignment-HPC-Failure-Prediction\notebooks
Project root: D:\_4TH_Year_2nd_Semester_Stuff\ML\Assignments\_Assignment\IT4060-ML-Assignment-HPC-Failure-Prediction
Interim directory: D:\_4TH_Year_2nd_Semester_Stuff\ML\Assignments\_Assignment\IT4060-ML-Assignment-HPC-Failure-Prediction\data\interim
Processed directory: D:\_4TH_Year_2nd_Semester_Stuff\ML\Assignments\_Assignment\IT4060-ML-Assignment-HPC-Failure-Prediction\data\processed


In [9]:
failure_df = pd.read_csv(
    failure_clean_path,
    parse_dates=['failure_start', 'failure_end'],
    low_memory=False,
)

usage_nodes_df = pd.read_csv(
    usage_nodes_clean_path,
    parse_dates=['usage_time', 'submit_time', 'dispatch_time', 'hour'],
    low_memory=False,
)

overview = pd.DataFrame([
    {'dataset': 'failure_clean', 'rows': len(failure_df), 'columns': failure_df.shape[1]},
    {'dataset': 'usage_node_events_clean', 'rows': len(usage_nodes_df), 'columns': usage_nodes_df.shape[1]},
])

display(overview)
display(usage_nodes_df[['usage_time', 'hour', 'node', 'batch_id', 'queue_wait_seconds', 'cpu_time_per_proc', 'status']].head())

,dataset,rows,columns
0,failure_clean,2122,28
1,usage_node_events_clean,1281928,33


,usage_time,hour,node,batch_id,queue_wait_seconds,cpu_time_per_proc,status
0,2003-12-26 12:32:30,2003-12-26 12:00:00,222,224261.0,66.0,7584.500000,aborted
1,2003-12-26 12:37:22,2003-12-26 12:00:00,249,224266.0,577.0,204467.000000,aborted
2,2003-12-26 12:41:37,2003-12-26 12:00:00,253,224269.0,836.0,206606.750000,aborted
3,2003-12-26 12:42:40,2003-12-26 12:00:00,146,224242.0,9.0,151.500000,aborted
4,2003-12-26 12:50:25,2003-12-26 12:00:00,3,224359.0,0.0,0.162504,finished


In [10]:
node_hour_features = usage_nodes_df.groupby(['node', 'hour'], as_index=False).agg(
    job_count=('batch_id', 'nunique'),
    unique_users=('user_id', 'nunique'),
    total_cpu_user_time=('cpu_user_time', 'sum'),
    total_cpu_system_time=('cpu_system_time', 'sum'),
    total_cpu_time=('total_time', 'sum'),
    avg_cpu_time_per_proc=('cpu_time_per_proc', 'mean'),
    avg_requested_procs=('num_procs', 'mean'),
    max_requested_procs=('num_procs', 'max'),
    avg_requested_procs_per_node=('requested_procs_per_node', 'mean'),
    avg_queue_wait_seconds=('queue_wait_seconds', 'mean'),
    median_queue_wait_seconds=('queue_wait_seconds', 'median'),
    avg_cpu_user_ratio=('cpu_user_ratio', 'mean'),
    weekend_jobs=('is_weekend', 'sum'),
    finished_jobs=('status_finished', 'sum'),
    aborted_jobs=('status_aborted', 'sum'),
    failed_jobs=('status_failed', 'sum'),
    killed_jobs=('status_killed', 'sum'),
    syskill_jobs=('status_syskill', 'sum'),
    missing_status_jobs=('status_missing', 'sum'),
    other_status_jobs=('status_other', 'sum'),
    dominant_hour_of_day=('hour_of_day', 'median'),
    dominant_day_of_week=('day_of_week', 'median'),
)

node_hour_features = node_hour_features.sort_values(['node', 'hour']).reset_index(drop=True)

lag_columns = ['job_count', 'unique_users', 'total_cpu_time', 'avg_queue_wait_seconds']
for column in lag_columns:
    node_hour_features[f'{column}_lag_1h'] = node_hour_features.groupby('node')[column].shift(1)
    node_hour_features[f'{column}_rolling_6h'] = (
        node_hour_features.groupby('node')[column]
        .transform(lambda values: values.shift(1).rolling(6, min_periods=1).mean())
    )

node_hour_features['cpu_time_share_system'] = np.where(
    node_hour_features['total_cpu_time'] > 0,
    node_hour_features['total_cpu_system_time'] / node_hour_features['total_cpu_time'],
    np.nan,
)
node_hour_features['aborted_job_ratio'] = np.where(
    node_hour_features['job_count'] > 0,
    node_hour_features['aborted_jobs'] / node_hour_features['job_count'],
    0,
)
node_hour_features['failed_job_ratio'] = np.where(
    node_hour_features['job_count'] > 0,
    node_hour_features['failed_jobs'] / node_hour_features['job_count'],
    0,
)

feature_summary = pd.DataFrame([
    {'metric': 'Node-hour rows before labeling', 'value': len(node_hour_features)},
    {'metric': 'Unique nodes', 'value': node_hour_features['node'].nunique()},
    {'metric': 'First hour', 'value': node_hour_features['hour'].min()},
    {'metric': 'Last hour', 'value': node_hour_features['hour'].max()},
])

display(feature_summary)
display(node_hour_features.head())

,metric,value
0,Node-hour rows before labeling,687814
1,Unique nodes,256
2,First hour,2003-12-26 12:00:00
3,Last hour,2005-10-21 20:00:00


,node,hour,job_count,unique_users,total_cpu_user_time,total_cpu_system_time,total_cpu_time,avg_cpu_time_per_proc,avg_requested_procs,max_requested_procs,avg_requested_procs_per_node,avg_queue_wait_seconds,median_queue_wait_seconds,avg_cpu_user_ratio,weekend_jobs,finished_jobs,aborted_jobs,failed_jobs,killed_jobs,syskill_jobs,missing_status_jobs,other_status_jobs,dominant_hour_of_day,dominant_day_of_week,job_count_lag_1h,job_count_rolling_6h,unique_users_lag_1h,unique_users_rolling_6h,total_cpu_time_lag_1h,total_cpu_time_rolling_6h,avg_queue_wait_seconds_lag_1h,avg_queue_wait_seconds_rolling_6h,cpu_time_share_system,aborted_job_ratio,failed_job_ratio
0,0,2003-12-26 13:00:00,4,2,376.575840,92.974048,469.549888,29.346868,4.0,4,4.0,1.5,1.0,0.344628,0,4,0,0,0,0,0,0,13.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.198007,0.0,0.0
1,0,2003-12-26 14:00:00,2,1,0.362096,0.956480,1.318576,0.164822,4.0,4,4.0,0.5,0.5,0.274664,0,2,0,0,0,0,0,0,14.0,4.0,4.0,4.000000,2.0,2.000000,469.549888,469.549888,1.5,1.5,0.725389,0.0,0.0
2,0,2003-12-26 15:00:00,2,2,0.441152,1.192672,1.633824,0.204228,4.0,4,4.0,1.0,1.0,0.266999,0,2,0,0,0,0,0,0,15.0,4.0,2.0,3.000000,1.0,1.500000,1.318576,235.434232,0.5,1.0,0.729988,0.0,0.0
3,0,2003-12-26 19:00:00,2,2,773.166896,520.478240,1293.645136,80.893142,6.0,8,4.0,1.0,1.0,0.428267,0,2,0,0,0,0,0,0,19.0,4.0,2.0,2.666667,2.0,1.666667,1.633824,157.500763,1.0,1.0,0.402335,0.0,0.0
4,0,2003-12-27 00:00:00,2,2,21.169824,15.473360,36.643184,4.580398,4.0,4,4.0,1.0,1.0,0.423685,2,2,0,0,0,0,0,0,0.0,5.0,2.0,2.500000,2.0,1.750000,1293.645136,441.536856,1.0,1.0,0.422271,0.0,0.0


In [11]:
HORIZONS_HOURS = [1, 6, 12, 24]

failure_events = failure_df[['nodenumz', 'failure_start']].dropna().copy()
failure_events = failure_events.rename(columns={'nodenumz': 'node', 'failure_start': 'failure_time'})
failure_events['node'] = failure_events['node'].astype(int)
failure_events = failure_events.sort_values(['node', 'failure_time'])

failure_time_map = {
    node: group['failure_time'].to_numpy(dtype='datetime64[ns]')
    for node, group in failure_events.groupby('node')
}

def attach_next_failure(group):
    group = group.sort_values('hour').copy()
    node = int(group['node'].iloc[0])
    failure_times = failure_time_map.get(node)
    if failure_times is None or len(failure_times) == 0:
        group['next_failure_time'] = pd.NaT
        group['hours_to_next_failure'] = np.nan
        for horizon in HORIZONS_HOURS:
            group[f'label_next_{horizon}h'] = 0
        return group

    hour_values = group['hour'].to_numpy(dtype='datetime64[ns]')
    positions = np.searchsorted(failure_times, hour_values, side='right')
    next_failure_values = np.full(len(group), np.datetime64('NaT', 'ns'))
    valid_positions = positions < len(failure_times)
    next_failure_values[valid_positions] = failure_times[positions[valid_positions]]

    group['next_failure_time'] = pd.to_datetime(next_failure_values)
    group['hours_to_next_failure'] = (
        group['next_failure_time'] - group['hour']
    ).dt.total_seconds() / 3600
    for horizon in HORIZONS_HOURS:
        group[f'label_next_{horizon}h'] = (
            group['hours_to_next_failure'].between(0, horizon, inclusive='both').fillna(False).astype(int)
        )
    return group

labeled_groups = []
for node, group in node_hour_features.groupby('node', sort=False):
    group = group.copy()
    group['node'] = node
    labeled_groups.append(attach_next_failure(group))

model_df = pd.concat(labeled_groups, ignore_index=True)

label_summary_rows = [
    {'metric': 'Node-hour rows', 'value': len(model_df)},
    {'metric': 'Rows with known next failure time', 'value': int(model_df['next_failure_time'].notna().sum())},
]
for horizon in HORIZONS_HOURS:
    label_summary_rows.append(
        {'metric': f'Positive labels in next {horizon} hours', 'value': int(model_df[f'label_next_{horizon}h'].sum())}
    )
label_summary = pd.DataFrame(label_summary_rows)

display(label_summary)
display(model_df[['node', 'hour', 'next_failure_time', 'hours_to_next_failure', 'label_next_1h', 'label_next_6h', 'label_next_12h', 'label_next_24h']].head())

,metric,value
0,Node-hour rows,687814
1,Rows with known next failure time,329646
2,Positive labels in next 1 hours,122
3,Positive labels in next 6 hours,701
4,Positive labels in next 12 hours,1374
5,Positive labels in next 24 hours,2820


,node,hour,next_failure_time,hours_to_next_failure,label_next_1h,label_next_6h,label_next_12h,label_next_24h
0,0,2003-12-26 13:00:00,2004-02-04 14:45:00,961.75,0,0,0,0
1,0,2003-12-26 14:00:00,2004-02-04 14:45:00,960.75,0,0,0,0
2,0,2003-12-26 15:00:00,2004-02-04 14:45:00,959.75,0,0,0,0
3,0,2003-12-26 19:00:00,2004-02-04 14:45:00,955.75,0,0,0,0
4,0,2003-12-27 00:00:00,2004-02-04 14:45:00,950.75,0,0,0,0


In [12]:
model_df.to_csv(feature_export_path, index=False, compression='gzip')

export_summary = pd.DataFrame([
    {'file_name': feature_export_path.name, 'rows': len(model_df), 'description': 'Processed node-hour feature table with multi-horizon failure labels'},
])

display(export_summary)
print(f'Processed feature table saved to: {feature_export_path}')

,file_name,rows,description
0,node_hour_features_multi_horizon.csv.gz,687814,Processed node-hour feature table with multi-h...


Processed feature table saved to: D:\_4TH_Year_2nd_Semester_Stuff\ML\Assignments\_Assignment\IT4060-ML-Assignment-HPC-Failure-Prediction\data\processed\node_hour_features_multi_horizon.csv.gz


## Next Step

Use the processed feature table for time-based train/test splitting, class-imbalance handling, and model training.